In [1]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Reading from CSV Files")
    .master("local[*]")
    .getOrCreate()
)

spark

In [5]:
df = spark.read.format("csv").load ("scratch/input/emp.csv")

In [6]:
df.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)



In [7]:
# Read a csv file into dataframe
df = spark.read.format("csv").option("header", True).option("inferSchema", True).load("scratch/input/emp.csv")

In [8]:
df.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- hire_date: timestamp (nullable = true)



In [9]:
df.show()

+-----------+-------------+-------------+---+------+------+-------------------+
|employee_id|department_id|         name|age|gender|salary|          hire_date|
+-----------+-------------+-------------+---+------+------+-------------------+
|          1|          101|     John Doe| 30|  Male| 50000|2015-01-01 00:00:00|
|          2|          101|   Jane Smith| 25|Female| 45000|2016-02-15 00:00:00|
|          3|          102|    Bob Brown| 35|  Male| 55000|2014-05-01 00:00:00|
|          4|          102|    Alice Lee| 28|Female| 48000|2017-09-30 00:00:00|
|          5|          103|    Jack Chan| 40|  Male| 60000|2013-04-01 00:00:00|
|          6|          103|    Jill Wong| 32|Female| 52000|2018-07-01 00:00:00|
|          7|          101|James Johnson| 42|  Male| 70000|2012-03-15 00:00:00|
|          8|          102|     Kate Kim| 29|Female| 51000|2019-10-01 00:00:00|
|          9|          103|      Tom Tan| 33|  Male| 58000|2016-06-01 00:00:00|
|         10|          104|     Lisa Lee

In [10]:
# Reading with Schema
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date"

df_schema = spark.read.format("csv").option("header",True).schema(_schema).load("scratch/input/emp.csv")


In [11]:
df_schema.printSchema() 

root
 |-- employee_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- hire_date: date (nullable = true)



In [12]:
df_schema.show()

+-----------+-------------+-------------+---+------+-------+----------+
|employee_id|department_id|         name|age|gender| salary| hire_date|
+-----------+-------------+-------------+---+------+-------+----------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|
|         10|          104|     Lisa Lee| 27|Female|47000.0|2018-08-01|
|         11|          104|   David Park| 38|  Male|65000.0|2015

In [ ]:
# Reading with Schema - Handle Bad records
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date, _corrupt_record string"

df_p = spark.read.format("csv").schema(_schema).option("header",True).load("scratch/input/emp_new.csv")


In [23]:
df_p.where("_corrupt_record is not null").show(truncate=False)

+-----------+-------------+-------------+---+------+-------+----------+--------------------------------------------+
|employee_id|department_id|name         |age|gender|salary |hire_date |_corrupt_record                             |
+-----------+-------------+-------------+---+------+-------+----------+--------------------------------------------+
|7          |101          |James Johnson|42 |Male  |null   |2012-03-15|007,101,James Johnson,42,Male,Low,2012-03-15|
|11         |104          |David Park   |38 |Male  |65000.0|null      |011,104,David Park,38,Male,65000,no date    |
+-----------+-------------+-------------+---+------+-------+----------+--------------------------------------------+



In [ ]:
# Reading with Schema - Handle Bad records _v2
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date, bad_record string"

df_p = spark.read.format("csv").schema(_schema).option("columnNameOfCorruptRecord","bad_record").option("header",True).load("scratch/input/emp_new.csv")


In [31]:
df_p.show(truncate=False)

+-----------+-------------+-------------+---+------+-------+----------+--------------------------------------------+
|employee_id|department_id|name         |age|gender|salary |hire_date |bad_record                                  |
+-----------+-------------+-------------+---+------+-------+----------+--------------------------------------------+
|1          |101          |John Doe     |30 |Male  |50000.0|2015-01-01|null                                        |
|2          |101          |Jane Smith   |25 |Female|45000.0|2016-02-15|null                                        |
|3          |102          |Bob Brown    |35 |Male  |55000.0|2014-05-01|null                                        |
|4          |102          |Alice Lee    |28 |Female|48000.0|2017-09-30|null                                        |
|5          |103          |Jack Chan    |40 |Male  |60000.0|2013-04-01|null                                        |
|6          |103          |Jill Wong    |32 |Female|52000.0|2018

In [ ]:
# Reading with Schema - Handle Bad records _v2 - droppedM Malformed records
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date"

df_m = spark.read.format("csv").schema(_schema).option("mode", "DROPMALFORMED").option("header",True).load("scratch/input/emp_new.csv")


In [30]:
df_m.show()

+-----------+-------------+-----------+---+------+-------+----------+
|employee_id|department_id|       name|age|gender| salary| hire_date|
+-----------+-------------+-----------+---+------+-------+----------+
|          1|          101|   John Doe| 30|  Male|50000.0|2015-01-01|
|          2|          101| Jane Smith| 25|Female|45000.0|2016-02-15|
|          3|          102|  Bob Brown| 35|  Male|55000.0|2014-05-01|
|          4|          102|  Alice Lee| 28|Female|48000.0|2017-09-30|
|          5|          103|  Jack Chan| 40|  Male|60000.0|2013-04-01|
|          6|          103|  Jill Wong| 32|Female|52000.0|2018-07-01|
|          8|          102|   Kate Kim| 29|Female|51000.0|2019-10-01|
|          9|          103|    Tom Tan| 33|  Male|58000.0|2016-06-01|
|         10|          104|   Lisa Lee| 27|Female|47000.0|2018-08-01|
|         12|          105| Susan Chen| 31|Female|54000.0|2017-02-15|
|         13|          106|  Brian Kim| 45|  Male|75000.0|2011-07-01|
|         14|       

In [ ]:
# Reading with Schema - Handle Bad records _v2 - FAILFAST - IF DATA NOT IN EXPECTED FORMAT, THROW ERROR
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date"

df_f = spark.read.format("csv").schema(_schema).option("mode", "FAILFAST").option("header",True).load("scratch/input/emp_new.csv")


In [34]:
df_f.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- hire_date: date (nullable = true)



In [ ]:
# BONUS TIP
# Multiple options - use

_options = {
    "header" : "true",
    "inferSchema" : "true",
    "mode" : "PERMISSIVE"
}

df = (spark.read.format("csv").options(**_options).load("scratch/input/emp.csv"))


In [36]:
df.show()

+-----------+-------------+-------------+---+------+------+-------------------+
|employee_id|department_id|         name|age|gender|salary|          hire_date|
+-----------+-------------+-------------+---+------+------+-------------------+
|          1|          101|     John Doe| 30|  Male| 50000|2015-01-01 00:00:00|
|          2|          101|   Jane Smith| 25|Female| 45000|2016-02-15 00:00:00|
|          3|          102|    Bob Brown| 35|  Male| 55000|2014-05-01 00:00:00|
|          4|          102|    Alice Lee| 28|Female| 48000|2017-09-30 00:00:00|
|          5|          103|    Jack Chan| 40|  Male| 60000|2013-04-01 00:00:00|
|          6|          103|    Jill Wong| 32|Female| 52000|2018-07-01 00:00:00|
|          7|          101|James Johnson| 42|  Male| 70000|2012-03-15 00:00:00|
|          8|          102|     Kate Kim| 29|Female| 51000|2019-10-01 00:00:00|
|          9|          103|      Tom Tan| 33|  Male| 58000|2016-06-01 00:00:00|
|         10|          104|     Lisa Lee

In [37]:
spark.stop()